In [1]:
import pandas as pd
import numpy as np

def create_advanced_features(df):
    print("Khởi chạy hệ thống Feature Engineering...")
    df_feat = df.copy()

    epsilon = 1e-5 # Hằng số siêu nhỏ để tránh lỗi chia cho 0 (Divide by Zero)

    # =====================================================================
    # NHÓM 1: TỶ LỆ SỨC KHỎE TÀI CHÍNH (Kết thúc bằng _A)
    # =====================================================================

    # 1. DTI (Debt-to-Income): Tỷ lệ nợ trên thu nhập. > 1 là báo động đỏ.
    df_feat['dti_ratio_A'] = (df_feat['totaldebt_9A'] / (df_feat['mainoccupationinc_384A'] + epsilon)).astype(np.float32)

    # 2. Credit Utilization: Tỷ lệ sử dụng hạn mức. Đo lường mức độ "vắt kiệt" tín dụng.
    df_feat['credit_utilization_A'] = (df_feat['currdebt_22A'] / (df_feat['credamount_770A'] + epsilon)).astype(np.float32)

    # 3. Settlement Ratio: Năng lực hoàn tất nợ trong quá khứ so với khoản vay mới.
    df_feat['settlement_to_loan_ratio_A'] = (df_feat['totalsettled_863A'] / (df_feat['credamount_770A'] + epsilon)).astype(np.float32)

    # =====================================================================
    # NHÓM 2: ĐỘNG LƯỢNG HÀNH VI / TỐC ĐỘ (Kết thúc bằng _A)
    # =====================================================================

    # 4. App Urgency Ratio: Tỷ trọng nộp hồ sơ tháng cuối so với lịch sử (Khát tiền ngắn hạn)
    df_feat['app_urgency_ratio_A'] = (df_feat['applications30d_658L'] / (df_feat['applicationcnt_361L'] + epsilon)).astype(np.float32)

    # 5. Query Momentum: Vận tốc tra cứu CIC. 1 tháng gần đây so với trung bình 6 tháng.
    avg_queries_per_month_6m = df_feat['days180_256L'] / 6.0
    df_feat['query_momentum_A'] = (df_feat['days30_165L'] / (avg_queries_per_month_6m + epsilon)).astype(np.float32)

    # =====================================================================
    # NHÓM 3: CỜ RỦI RO / LỌC LOGIC (Tiền tố is_/has_ và kết thúc bằng _L)
    # =====================================================================

    # 6. Lịch sử trễ hạn nghiêm trọng (Từng quá hạn > 30 ngày)
    df_feat['has_serious_default_L'] = (df_feat['maxdpdlast24m_143P'] > 30).astype(np.int8)

    # 7. Nhóm nhân khẩu học rủi ro (Kết hợp EDA: Tư nhân + Ly hôn/Sống thử)
    cond_income = df_feat['incometype_1044T'] == 'PRIVATE_SECTOR_EMPLOYEE'
    cond_family = df_feat['familystate_447L'].isin(['DIVORCED', 'LIVING_WITH_PARTNER'])
    df_feat['is_high_risk_profile_L'] = (cond_income & cond_family).astype(np.int8)

    # 8. Cờ "Hoảng loạn tín dụng": Vừa nộp nhiều đơn, vừa bị CIC tra cứu mạnh trong 30 ngày
    cond_apps = df_feat['applications30d_658L'] > 1
    cond_queries = df_feat['days30_165L'] > 2
    df_feat['is_credit_hungry_L'] = (cond_apps & cond_queries).astype(np.int8)

    print(f"Đã tạo thành công 8 đặc trưng mới. Tổng số cột hiện tại: {df_feat.shape[1]}")
    return df_feat
df = pd.read_parquet("/data/processed/credit_data_processed_v2.parquet")
# Áp dụng hàm vào bộ dữ liệu
df = create_advanced_features(df)

# Kiểm tra danh sách các cột mới tạo ra
new_cols = [col for col in df.columns if col.endswith('_A') or col.endswith('_L')]
print("\nDanh sách các biến tài chính và cờ logic (Naming Convention):")
print([col for col in new_cols if col not in ['totaldebt_9A', 'currdebt_22A', 'credamount_770A', 'mainoccupationinc_384A', 'totalsettled_863A', 'applicationcnt_361L', 'applications30d_658L', 'days30_165L', 'days180_256L', 'days120_123L', 'days90_310L', 'days360_512L', 'sex_738L', 'familystate_447L', 'empl_employedtotal_800L']]) # Lọc loại bỏ cột gốc để in ra cột mới

Khởi chạy hệ thống Feature Engineering...
Đã tạo thành công 8 đặc trưng mới. Tổng số cột hiện tại: 57

Danh sách các biến tài chính và cờ logic (Naming Convention):
['dti_ratio_A', 'credit_utilization_A', 'settlement_to_loan_ratio_A', 'app_urgency_ratio_A', 'query_momentum_A', 'has_serious_default_L', 'is_high_risk_profile_L', 'is_credit_hungry_L']


In [2]:
print(f"Shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")
print(f"\nTarget distribution:\n{df['target'].value_counts()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nData types:\n{df.dtypes}")

Shape: (522596, 57)

Columns:
['case_id', 'date_decision', 'MONTH', 'WEEK_NUM', 'target', 'maxdpdfrom6mto36m_3546853P', 'maxdpdlast12m_727P', 'maxdpdlast24m_143P', 'maxdpdlast3m_392P', 'maxdpdlast6m_474P', 'numinstls_657L', 'numinstlsallpaid_934L', 'pctinstlsallpaidlat10d_839L', 'totalsettled_863A', 'totaldebt_9A', 'currdebt_22A', 'credamount_770A', 'mobilephncnt_593L', 'homephncnt_628L', 'numactivecreds_622L', 'applicationcnt_361L', 'applications30d_658L', 'applicationscnt_1086L', 'applicationscnt_464L', 'applicationscnt_629L', 'avgdbddpdlast24m_3658932P', 'amtinstpaidbefduel24m_4187115A', 'maxdbddpdlast1m_3658939P', 'maxdbddpdtollast12m_3658940P', 'maxdbddpdtollast6m_4187119P', 'numinstpaidlate1d_3546852L', 'numberofqueries_373L', 'days120_123L', 'days180_256L', 'days30_165L', 'days90_310L', 'days360_512L', 'sex_738L', 'education_927M', 'incometype_1044T', 'familystate_447L', 'empl_employedtotal_800L', 'language1_981M', 'mainoccupationinc_384A', 'age', 'maritalst_385M', 'requesttype_